In [ ]:
import pandas as pd
import re
from difflib import SequenceMatcher

# Define the set of known placeholders.
PLACEHOLDERS = {"NAME", "LOCATION", "EMAIL", "JOB", "MASKEDNUMBER",
                "URL", "USERNAME", "IP", "PASSWORD", "GENDER",
                "CREDITCARDNUMBER", "USERAGENT", "IBAN"}

def extract_masked_tokens(text, return_positions=False):
    """
    Extracts all masked tokens that appear inside square brackets.
    For example, "[ name ]" will yield the token "name".
    If return_positions is True, returns a list of tuples (token, start, end).
    Otherwise, returns just the list of token strings.
    """
    pattern = re.compile(r'\[\s*(' + '|'.join(PLACEHOLDERS) + r')\s*\]', re.IGNORECASE)
    if return_positions:
        return [(m.group(1), m.start(), m.end()) for m in pattern.finditer(text)]
    else:
        return [m.group(1) for m in pattern.finditer(text)]

def map_gold_span_to_pred(gold, pred, g_start, g_end):
    """
    Maps a span (g_start, g_end) from the gold text to the corresponding span in the predicted text.
    Uses SequenceMatcher to align the two strings. Returns a tuple (pred_start, pred_end).
    If mapping fails, returns (None, None).
    """
    sm = SequenceMatcher(None, gold, pred)
    opcodes = sm.get_opcodes()
    pred_start = None
    pred_end = None
    for tag, i1, i2, j1, j2 in opcodes:
        # Map the start of the gold token.
        if g_start >= i1 and g_start < i2:
            if tag in ("equal", "replace"):
                pred_start = j1 + (g_start - i1)
            else:
                pred_start = j1
        # Map the end of the gold token.
        if g_end > i1 and g_end <= i2:
            if tag in ("equal", "replace"):
                pred_end = j1 + (g_end - i1)
            else:
                pred_end = j1
        if pred_start is not None and pred_end is not None:
            break
    return pred_start, pred_end

def analyze_row_aligned(original, gold, pred):
    """
    For each masked token in the gold (labeled_text), the function maps its span to the predicted
    text and then compares the tokens.
      - If the token in the predicted text (after stripping brackets and extra spaces) matches
        the gold token (ignoring case), it is counted as correct.
      - Otherwise, it is counted as a misplaced token.
      - If no valid mapping is found, the token is counted as missing.
    Additionally, any extra predicted masked tokens that were not aligned with a gold token are
    flagged as false positives.
    
    Returns:
      - metrics: dictionary with counts for correct, misplaced, missing, and false_positive.
      - examples: dictionary of examples for error cases.
    """
    gold_tokens = extract_masked_tokens(gold, return_positions=True)
    metrics = {"correct": 0, "misplaced": 0, "missing": 0, "false_positive": 0}
    examples = {"misplaced": []}
    
    # Process each gold token.
    for g_token, g_start, g_end in gold_tokens:
        p_start, p_end = map_gold_span_to_pred(gold, pred, g_start, g_end)
        if p_start is None or p_end is None or p_start < 0 or p_end > len(pred):
            metrics["missing"] += 1
            examples["misplaced"].append({
                "gold": g_token,
                "predicted": None,
                "gold_span": (g_start, g_end),
                "pred_span": None,
                "note": "Mapping not found"
            })
        else:
            # Extract the predicted token and clean by removing brackets.
            pred_token = pred[p_start:p_end]
            pred_token_clean = re.sub(r'[\[\]]', '', pred_token).strip()
            if pred_token_clean.lower() != g_token.lower():
                metrics["misplaced"] += 1
                examples["misplaced"].append({
                    "gold": g_token,
                    "predicted": pred_token_clean,
                    "gold_span": (g_start, g_end),
                    "pred_span": (p_start, p_end),
                    "note": "Mismatch token"
                })
            else:
                metrics["correct"] += 1

    # Identify extra predicted tokens that were not aligned from any gold token.
    pred_tokens = extract_masked_tokens(pred, return_positions=True)
    mapped_spans = []
    for g_token, g_start, g_end in gold_tokens:
        p_span = map_gold_span_to_pred(gold, pred, g_start, g_end)
        if p_span[0] is not None and p_span[1] is not None:
            mapped_spans.append(p_span)
    for p_token, p_start, p_end in pred_tokens:
        if not any(p_start == ms[0] and p_end == ms[1] for ms in mapped_spans):
            metrics["false_positive"] += 1
            examples["misplaced"].append({
                "gold": None,
                "predicted": p_token,
                "gold_span": None,
                "pred_span": (p_start, p_end),
                "note": "Extra predicted token"
            })
            
    return metrics, examples

def analyze_masking(file_path, original_col="original_text", gold_col="labeled_text", pred_col="masked_text"):
    """
    Reads a CSV file (here, dataset_1.csv) that contains three columns:
      - original_text: the unmasked original text,
      - labeled_text: the gold masked text (with placeholders inside square brackets),
      - masked_text: the predicted masked text.
    
    For each row, it computes masking metrics by comparing the gold and predicted tokens.
    Finally, it prints per-row metrics, aggregate counts, and evaluation metrics (Precision, Recall, F1).
    """
    df = pd.read_csv(file_path)
    print("Available columns in the DataFrame:", df.columns.tolist())
    
    all_metrics = {"correct": 0, "misplaced": 0, "missing": 0, "false_positive": 0}
    all_examples = {"misplaced": []}
    detailed_results = []
    
    # Process each row.
    for idx, row in df.iterrows():
        metrics, examples = analyze_row_aligned(
            row[original_col],
            row[gold_col],
            row[pred_col]
        )
        detailed_results.append(metrics)
        for key in all_metrics:
            all_metrics[key] += metrics.get(key, 0)
        for ex in examples["misplaced"]:
            ex["row"] = idx
            ex["original"] = row[original_col]
            ex["labeled_text"] = row[gold_col]
            ex["masked_text"] = row[pred_col]
            all_examples["misplaced"].append(ex)
    
    metrics_df = pd.DataFrame(detailed_results)
    print("\nPer-row Masking Metrics:")
    print(metrics_df)
    
    print("\nAggregate Masking Metrics:")
    for key, value in all_metrics.items():
        print(f"{key}: {value}")
    
    # Compute evaluation metrics.
    TP = all_metrics["correct"]
    FN = all_metrics["missing"] + all_metrics["misplaced"]
    FP = all_metrics["false_positive"] + all_metrics["misplaced"]
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    F1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print("\nEvaluation Metrics:")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1 Score:  {F1:.3f}")
    
    print("\nExamples of Misplaced Tokens:")
    if all_examples["misplaced"]:
        for ex in all_examples["misplaced"]:
            print(f"Row {ex['row']} | Original: {ex['original']}")
            print(f"Labeled: {ex['labeled_text']}")
            print(f"Masked:  {ex['masked_text']}")
            print(f"Note: {ex.get('note', '')}")
            print(f"Gold: '{ex['gold']}' at {ex.get('gold_span')}, Predicted: '{ex.get('predicted')}' at {ex.get('pred_span')}\n")
    else:
        print("No misplaced tokens found.")
    
    return all_metrics, metrics_df, all_examples

if __name__ == "__main__":
    analyze_masking("dataset_1.csv")


Available columns: ['original_text', 'labeled_text', 'masked_text']

Aggregate Token-Level Masking Metrics:
  correct:        8138
  misplaced:      98
  missing:        3
  false_positive: 86

Overall Evaluation Metrics:
  Precision: 0.978
  Recall:    0.988
  F1 Score:  0.983

Per-Label Evaluation Metrics:
Label: NAME
  Gold Count:      5640
  Predicted Count: 5639
  Precision: 0.99  Recall: 1.00  F1: 0.99

Label: JOB
  Gold Count:      346
  Predicted Count: 356
  Precision: 0.92  Recall: 0.97  F1: 0.95

Label: LOCATION
  Gold Count:      1179
  Predicted Count: 1176
  Precision: 0.95  Recall: 0.97  F1: 0.96

Label: PASSWORD
  Gold Count:      35
  Predicted Count: 23
  Precision: 0.49  Recall: 0.54  F1: 0.51

Label: EMAIL
  Gold Count:      795
  Predicted Count: 796
  Precision: 1.00  Recall: 1.00  F1: 1.00

Label: MASKEDNUMBER
  Gold Count:      161
  Predicted Count: 167
  Precision: 0.93  Recall: 0.97  F1: 0.95

Label: NATHEN
  Gold Count:      0
  Predicted Count: 1
  Precisio

In [25]:
import pandas as pd
import re
from difflib import SequenceMatcher

# Allowed labels for evaluation
ALLOWED_LABELS =  {"NAME", "LOCATION", "EMAIL", "JOB", "MASKEDNUMBER",
                "URL", "USERNAME", "IP", "PASSWORD", "GENDER",
                "CREDITCARDNUMBER", "USERAGENT", "IBAN"}

def extract_masked_tokens(text, return_positions=False):
    """
    Extracts masked tokens that appear inside square brackets.
    Only tokens that match one of the allowed labels are returned.
    For example, "[ name ]" yields "NAME".
    If return_positions is True, returns a list of tuples (token, start, end),
    otherwise returns a list of token strings.
    """
    pattern = re.compile(r'\[\s*(' + '|'.join(ALLOWED_LABELS) + r')\s*\]', re.IGNORECASE)
    if return_positions:
        return [(m.group(1).upper().strip(), m.start(), m.end()) for m in pattern.finditer(text)]
    else:
        return [m.group(1).upper().strip() for m in pattern.finditer(text)]

def map_gold_span_to_pred(gold, pred, g_start, g_end):
    """
    Maps a span (g_start, g_end) from the gold text to the corresponding span in the predicted text.
    Uses SequenceMatcher to align the two strings.
    Returns a tuple (pred_start, pred_end); if mapping fails, returns (None, None).
    """
    sm = SequenceMatcher(None, gold, pred)
    opcodes = sm.get_opcodes()
    pred_start = None
    pred_end = None
    for tag, i1, i2, j1, j2 in opcodes:
        if g_start >= i1 and g_start < i2:
            if tag in ("equal", "replace"):
                pred_start = j1 + (g_start - i1)
            else:
                pred_start = j1
        if g_end > i1 and g_end <= i2:
            if tag in ("equal", "replace"):
                pred_end = j1 + (g_end - i1)
            else:
                pred_end = j1
        if pred_start is not None and pred_end is not None:
            break
    return pred_start, pred_end

def analyze_row_aligned(original, gold, pred):
    """
    For a single sentence (row), this function performs token-level evaluation.
    
    For each gold token (from the allowed labels):
      - It maps its span in the gold text to the predicted text.
      - If mapping fails, the token is marked as missing (a false negative for that label).
      - If mapping succeeds, the predicted token is cleaned (brackets removed)
        and compared to the gold token:
          • If they match (case-insensitively), it counts as correct (TP for that label).
          • Otherwise, it's a mismatch (counted as a false negative for the gold label and
            a false positive for the predicted label).
    
    Extra predicted tokens (i.e. masked tokens not aligned to any gold token) are flagged as false positives.
    
    Returns:
      overall_metrics: dict with token-level counts: correct, misplaced, missing, false_positive.
      per_label_counts: dict mapping each allowed label to counts {TP, FN, FP}.
      examples: dict containing examples of errors.
    """
    gold_tokens = extract_masked_tokens(gold, return_positions=True)
    overall_metrics = {"correct": 0, "misplaced": 0, "missing": 0, "false_positive": 0}
    examples = {"misplaced": []}
    # Initialize per-label counts.
    per_label_counts = {label: {"TP": 0, "FN": 0, "FP": 0} for label in ALLOWED_LABELS}
    
    mapped_spans = []
    for g_token, g_start, g_end in gold_tokens:
        gold_label = g_token  # Already upper-case and stripped.
        p_start, p_end = map_gold_span_to_pred(gold, pred, g_start, g_end)
        if p_start is None or p_end is None or p_start < 0 or p_end > len(pred):
            overall_metrics["missing"] += 1
            per_label_counts[gold_label]["FN"] += 1
            examples["misplaced"].append({
                "gold": gold_label,
                "predicted": None,
                "gold_span": (g_start, g_end),
                "pred_span": None,
                "note": "Mapping not found"
            })
        else:
            pred_token = pred[p_start:p_end]
            pred_token_clean = re.sub(r'[\[\]]', '', pred_token).upper().strip()
            if pred_token_clean == gold_label:
                overall_metrics["correct"] += 1
                per_label_counts[gold_label]["TP"] += 1
            else:
                overall_metrics["misplaced"] += 1
                per_label_counts[gold_label]["FN"] += 1
                if pred_token_clean in ALLOWED_LABELS:
                    per_label_counts[pred_token_clean]["FP"] += 1
                examples["misplaced"].append({
                    "gold": gold_label,
                    "predicted": pred_token_clean,
                    "gold_span": (g_start, g_end),
                    "pred_span": (p_start, p_end),
                    "note": "Mismatch token"
                })
            mapped_spans.append((p_start, p_end))
    
    # Count extra predicted tokens not mapped from any gold token.
    pred_tokens = extract_masked_tokens(pred, return_positions=True)
    for p_token, p_start, p_end in pred_tokens:
        if not any(p_start == ms[0] and p_end == ms[1] for ms in mapped_spans):
            overall_metrics["false_positive"] += 1
            predicted_label = p_token
            if predicted_label in ALLOWED_LABELS:
                per_label_counts[predicted_label]["FP"] += 1
            examples["misplaced"].append({
                "gold": None,
                "predicted": predicted_label,
                "gold_span": None,
                "pred_span": (p_start, p_end),
                "note": "Extra predicted token"
            })
    
    return overall_metrics, per_label_counts, examples

def analyze_masking(file_path, original_col="original_text", gold_col="labeled_text", pred_col="masked_text"):
    """
    Reads the CSV file (with columns: original_text, labeled_text, masked_text) and evaluates:
      1. Token-level (aggregate) metrics across all sentences.
      2. Per-label evaluation for only the allowed labels.
      3. Sentence-level evaluation: a sentence is considered correct if the ordered list of
         allowed masked tokens (from gold and predicted texts) match exactly.
    
    Note: The token-level metrics (correct, misplaced, etc.) count masked tokens—not sentences.
          In a test set with ~8000 sentences, the total masked token count may differ from 8000.
    
    Prints:
      - Aggregate token-level masking metrics and overall evaluation (Precision, Recall, F1).
      - Per-label evaluation metrics.
      - Sentence-level evaluation (total sentences, correct sentences, sentence accuracy).
      - Examples of error cases.
    """
    df = pd.read_csv(file_path)
    print("Available columns:", df.columns.tolist())
    
    total_sentences = len(df)
    correct_sentence_count = 0
    aggregate_metrics = {"correct": 0, "misplaced": 0, "missing": 0, "false_positive": 0}
    aggregate_per_label = {label: {"TP": 0, "FN": 0, "FP": 0} for label in ALLOWED_LABELS}
    sentence_details = []  # Per-sentence token-level details.
    all_examples = {"misplaced": []}
    
    for idx, row in df.iterrows():
        gold_text = row[gold_col]
        pred_text = row[pred_col]
        
        # Token-level evaluation.
        overall_metrics, per_label_counts, examples = analyze_row_aligned(row[original_col], gold_text, pred_text)
        sentence_details.append(overall_metrics)
        for key in aggregate_metrics:
            aggregate_metrics[key] += overall_metrics.get(key, 0)
        for label in ALLOWED_LABELS:
            aggregate_per_label[label]["TP"] += per_label_counts[label]["TP"]
            aggregate_per_label[label]["FN"] += per_label_counts[label]["FN"]
            aggregate_per_label[label]["FP"] += per_label_counts[label]["FP"]
        for ex in examples["misplaced"]:
            ex["row"] = idx
            ex["original"] = row[original_col]
            ex["labeled_text"] = gold_text
            ex["masked_text"] = pred_text
            all_examples["misplaced"].append(ex)
        
        # Sentence-level evaluation: compare ordered list of allowed tokens.
        gold_tokens_list = extract_masked_tokens(gold_text, return_positions=False)
        pred_tokens_list = extract_masked_tokens(pred_text, return_positions=False)
        if gold_tokens_list == pred_tokens_list:
            correct_sentence_count += 1
    
    # Compute overall token-level metrics.
    TP = aggregate_metrics["correct"]
    FN = aggregate_metrics["missing"] + aggregate_metrics["misplaced"]
    FP = aggregate_metrics["false_positive"] + aggregate_metrics["misplaced"]
    precision_overall = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall_overall = TP / (TP + FN) if (TP + FN) > 0 else 0
    F1_overall = 2 * precision_overall * recall_overall / (precision_overall + recall_overall) if (precision_overall + recall_overall) > 0 else 0
    
    print("\nAggregate Token-Level Masking Metrics:")
    print(f"  correct:        {aggregate_metrics['correct']}")
    print(f"  misplaced:      {aggregate_metrics['misplaced']}")
    print(f"  missing:        {aggregate_metrics['missing']}")
    print(f"  false_positive: {aggregate_metrics['false_positive']}")
    
    print("\nOverall Evaluation Metrics (Token-Level):")
    print(f"  Precision: {precision_overall:.3f}")
    print(f"  Recall:    {recall_overall:.3f}")
    print(f"  F1 Score:  {F1_overall:.3f}")
    
    print("\nPer-Label Evaluation Metrics:")
    for label in sorted(ALLOWED_LABELS):
        counts = aggregate_per_label[label]
        gold_count = counts["TP"] + counts["FN"]
        pred_count = counts["TP"] + counts["FP"]
        precision = counts["TP"] / (counts["TP"] + counts["FP"]) if (counts["TP"] + counts["FP"]) > 0 else 0
        recall = counts["TP"] / (counts["TP"] + counts["FN"]) if (counts["TP"] + counts["FN"]) > 0 else 0
        F1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        print(f"Label: {label}")
        print(f"  Gold Count:      {gold_count}")
        print(f"  Predicted Count: {pred_count}")
        print(f"  Precision: {precision:.2f}  Recall: {recall:.2f}  F1: {F1:.2f}\n")
    
    # Sentence-level evaluation.
    sentence_accuracy = correct_sentence_count / total_sentences if total_sentences > 0 else 0
    print("Sentence-Level Evaluation:")
    print(f"  Total sentences:   {total_sentences}")
    #print neew line
    print("\n")
    
    print(f"  Correct sentences: {correct_sentence_count}")
    print("\n")
    print(f"  Sentence Accuracy: {sentence_accuracy:.3f}")
    
    print("\nExamples of Errors:")
    for ex in all_examples["misplaced"]:
        print(f"Row {ex['row']}: {ex}")
    
    return aggregate_metrics, sentence_details, aggregate_per_label, all_examples

if __name__ == "__main__":
    analyze_masking("dataset_1.csv")


Available columns: ['original_text', 'labeled_text', 'masked_text']

Aggregate Token-Level Masking Metrics:
  correct:        8286
  misplaced:      144
  missing:        5
  false_positive: 83

Overall Evaluation Metrics (Token-Level):
  Precision: 0.973
  Recall:    0.982
  F1 Score:  0.978

Per-Label Evaluation Metrics:
Label: CREDITCARDNUMBER
  Gold Count:      7
  Predicted Count: 22
  Precision: 0.32  Recall: 1.00  F1: 0.48

Label: EMAIL
  Gold Count:      795
  Predicted Count: 795
  Precision: 1.00  Recall: 1.00  F1: 1.00

Label: GENDER
  Gold Count:      23
  Predicted Count: 22
  Precision: 0.91  Recall: 0.87  F1: 0.89

Label: IBAN
  Gold Count:      2
  Predicted Count: 3
  Precision: 0.67  Recall: 1.00  F1: 0.80

Label: IP
  Gold Count:      51
  Predicted Count: 57
  Precision: 0.82  Recall: 0.92  F1: 0.87

Label: JOB
  Gold Count:      346
  Predicted Count: 332
  Precision: 0.98  Recall: 0.94  F1: 0.96

Label: LOCATION
  Gold Count:      1179
  Predicted Count: 1165
  Pr

In [29]:
import pandas as pd
import re
from difflib import SequenceMatcher

# Define the set of known placeholders.
PLACEHOLDERS = {"NAME", "LOCATION", "EMAIL", "JOB", "MASKEDNUMBER",
                "URL", "USERNAME", "IP", "PASSWORD", "GENDER",
                "CREDITCARDNUMBER", "USERAGENT", "IBAN"}

def extract_masked_tokens(text, return_positions=False):
    """
    Extracts all masked tokens that match known placeholders.
    If return_positions is True, returns a list of tuples (token, start, end).
    Otherwise, returns just the list of token strings.
    """
    pattern = re.compile(r'\b(' + '|'.join(PLACEHOLDERS) + r')\b', re.IGNORECASE)
    if return_positions:
        return [(m.group(), m.start(), m.end()) for m in pattern.finditer(text)]
    else:
        return pattern.findall(text)

def map_gold_span_to_pred(gold, pred, g_start, g_end):
    """
    Maps a span (g_start, g_end) from the gold text to the corresponding span in the predicted text
    using SequenceMatcher to align the two strings.
    Returns a tuple (pred_start, pred_end). If mapping fails, returns (None, None).
    """
    sm = SequenceMatcher(None, gold, pred)
    opcodes = sm.get_opcodes()
    pred_start = None
    pred_end = None
    for tag, i1, i2, j1, j2 in opcodes:
        if g_start >= i1 and g_start < i2:
            pred_start = j1 + (g_start - i1) if tag in ("equal", "replace") else j1
        if g_end > i1 and g_end <= i2:
            pred_end = j1 + (g_end - i1) if tag in ("equal", "replace") else j1
        if pred_start is not None and pred_end is not None:
            break
    return pred_start, pred_end

def find_overlapping_token(pred_text, mapped_span, valid_labels):
    """
    Given the predicted text, a mapped span (p_start, p_end), and a set of valid labels,
    search among all tokens (extracted via regex) for one whose span overlaps with mapped_span.
    Returns a tuple (token, (start, end)) if found; otherwise (None, None).
    """
    p_tokens = extract_masked_tokens(pred_text, return_positions=True)
    p_start, p_end = mapped_span
    for token, start, end in p_tokens:
        # Check if there is any overlap
        if not (end < p_start or start > p_end):
            if token.lower() in valid_labels:
                return token, (start, end)
    return None, None

def analyze_row_aligned(original, gold, pred):
    """
    For each gold token, maps its span to the predicted text and then:
      - If mapping fails, counts as missing.
      - Otherwise, extracts a raw predicted token from the mapped span.
        If that token isn’t valid (i.e. not one of our placeholders), it searches the
        predicted text (within the mapped region) for a valid token.
      - If a valid predicted token is found, compares it with the gold token:
          * If they match (ignoring case), counts as correct.
          * Otherwise, counts as misplaced.
    Extra predicted tokens (those that are not mapped to any gold token) are counted as false positives.
    
    Returns:
      - metrics: dictionary with overall counts.
      - examples: dictionary with examples of error cases.
      - label_metrics: dictionary with per-label counts.
    """
    gold_tokens = extract_masked_tokens(gold, return_positions=True)
    metrics = {"correct": 0, "misplaced": 0, "missing": 0, "false_positive": 0}
    examples = {"misplaced": []}
    # Initialize per-label counts.
    label_metrics = { label: {"gold": 0, "predicted": 0, "correct": 0,
                               "misplaced": 0, "missing": 0, "false_positive": 0}
                      for label in PLACEHOLDERS }
    
    valid_labels = {p.lower() for p in PLACEHOLDERS}
    
    for idx, (g_token, g_start, g_end) in enumerate(gold_tokens):
        token_upper = g_token.upper()
        label_metrics[token_upper]["gold"] += 1
        
        p_start, p_end = map_gold_span_to_pred(gold, pred, g_start, g_end)
        if p_start is None or p_end is None or p_start < 0 or p_end > len(pred):
            metrics["missing"] += 1
            label_metrics[token_upper]["missing"] += 1
            examples["misplaced"].append({
                "gold": g_token,
                "predicted": None,
                "gold_span": (g_start, g_end),
                "pred_span": None,
                "note": "Mapping not found"
            })
            continue
        
        # Extract the raw predicted token from the mapped span.
        raw_pred = pred[p_start:p_end].strip()
        # If raw predicted token is not a valid label, try to find a valid candidate in the vicinity.
        if raw_pred.lower() not in valid_labels:
            candidate, candidate_span = find_overlapping_token(pred, (p_start, p_end), valid_labels)
            if candidate is not None:
                pred_token = candidate
                pred_span = candidate_span
            else:
                pred_token = raw_pred
                pred_span = (p_start, p_end)
        else:
            pred_token = raw_pred
            pred_span = (p_start, p_end)
        
        # If after recovery, the predicted token is still not valid, count as missing.
        if pred_token.lower() not in valid_labels:
            metrics["missing"] += 1
            label_metrics[token_upper]["missing"] += 1
            examples["misplaced"].append({
                "gold": g_token,
                "predicted": pred_token,
                "gold_span": (g_start, g_end),
                "pred_span": pred_span,
                "note": "No valid label found near mapped span"
            })
        else:
            # Valid predicted token found: count it for that label.
            pred_label = pred_token.upper()
            label_metrics[pred_label]["predicted"] += 1
            if pred_label == token_upper:
                metrics["correct"] += 1
                label_metrics[token_upper]["correct"] += 1
            else:
                metrics["misplaced"] += 1
                label_metrics[token_upper]["misplaced"] += 1
                examples["misplaced"].append({
                    "gold": g_token,
                    "predicted": pred_token,
                    "gold_span": (g_start, g_end),
                    "pred_span": pred_span,
                    "note": "Wrong label"
                })
    
    # Identify extra predicted tokens not mapped from any gold token.
    pred_tokens = extract_masked_tokens(pred, return_positions=True)
    mapped_spans = []
    for (g_token, g_start, g_end) in gold_tokens:
        p_span = map_gold_span_to_pred(gold, pred, g_start, g_end)
        if p_span[0] is not None and p_span[1] is not None:
            mapped_spans.append(p_span)
    for p_token, p_start, p_end in pred_tokens:
        # If no mapped span overlaps this predicted token, count it as an extra.
        if not any(not (p_end < m[0] or p_start > m[1]) for m in mapped_spans):
            # Only count if the token is valid.
            if p_token.lower() in valid_labels:
                metrics["false_positive"] += 1
                extra_label = p_token.upper()
                label_metrics[extra_label]["false_positive"] += 1
                examples["misplaced"].append({
                    "gold": None,
                    "predicted": p_token,
                    "gold_span": None,
                    "pred_span": (p_start, p_end),
                    "note": "Extra predicted token"
                })
    
    return metrics, examples, label_metrics

def analyze_masking(file_path, original_col=None, gold_col=None, pred_col=None):
    """
    Reads a CSV (or pickle) file containing evaluation results with columns:
      - Original Text,
      - Labeled Text (gold masked template), and
      - Masked Text (predicted masked template).
    Defaults: original_text, labeled_text, masked_text.
    Prints per-row metrics, per-label counts, and overall evaluation metrics (Precision, Recall, F1, Accuracy)
    along with per-label evaluation metrics.
    """
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_pickle(file_path)
    
    if hasattr(df, "dataset"):
        df = df.dataset

    print("Available columns in the DataFrame:", df.columns.tolist())
    
    if original_col is None:
        if "original_text" in df.columns:
            original_col = "original_text"
        else:
            raise ValueError("Could not determine original text column. Available columns: " + str(df.columns.tolist()))
    
    if gold_col is None:
        if "labeled_text" in df.columns:
            gold_col = "labeled_text"
        else:
            raise ValueError("Could not determine gold masked text column. Available columns: " + str(df.columns.tolist()))
    
    if pred_col is None:
        if "masked_text" in df.columns:
            pred_col = "masked_text"
        else:
            raise ValueError("Could not determine predicted masked text column. Available columns: " + str(df.columns.tolist()))
    
    required_columns = [original_col, gold_col, pred_col]
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"DataFrame does not contain the required columns: {required_columns}")

    all_metrics = {"correct": 0, "misplaced": 0, "missing": 0, "false_positive": 0}
    all_examples = {"misplaced": []}
    detailed_results = []
    overall_label_metrics = { label: {"gold": 0, "predicted": 0, "correct": 0,
                                      "misplaced": 0, "missing": 0, "false_positive": 0}
                              for label in PLACEHOLDERS }
    
    for idx, row in df.iterrows():
        metrics, examples, row_label_metrics = analyze_row_aligned(
            row[original_col],
            row[gold_col],
            row[pred_col]
        )
        detailed_results.append(metrics)
        for key in all_metrics:
            all_metrics[key] += metrics.get(key, 0)
        for ex in examples["misplaced"]:
            ex["row"] = idx
            ex["original"] = row[original_col]
            ex["labeled_text"] = row[gold_col]
            ex["masked_text"] = row[pred_col]
            all_examples["misplaced"].append(ex)
        for label in overall_label_metrics:
            for key in overall_label_metrics[label]:
                overall_label_metrics[label][key] += row_label_metrics[label][key]
    
    metrics_df = pd.DataFrame(detailed_results)
    print("\nPer-row masking metrics:")
    print(metrics_df)
    
    print("\nAggregate masking metrics:")
    for key, value in all_metrics.items():
        print(f"{key}: {value}")
    
    note_counts = {"No valid label found near mapped span": 0, "Extra predicted token": 0}
    for ex in all_examples["misplaced"]:
        note = ex.get("note", "")
        if note in note_counts:
            note_counts[note] += 1

    print("\nCount of tokens with note 'No valid label found near mapped span':", 
          note_counts["No valid label found near mapped span"])
    print("Count of tokens with note 'Extra predicted token':", 
          note_counts["Extra predicted token"])
    
    print("\nExamples of Misplaced Tokens:")
    if all_examples["misplaced"]:
        for ex in all_examples["misplaced"]:
            print(f"Row {ex['row']} | Original Text:  {ex['original']}")
            print(f"Row {ex['row']} | Labeled Text:   {ex['labeled_text']}")
            print(f"Row {ex['row']} | Masked Text:    {ex['masked_text']}")
            print(f"  Note: {ex.get('note', '')}")
            print(f"  Gold: '{ex['gold']}' at {ex['gold_span']} vs Predicted: '{ex['predicted']}' at {ex['pred_span']}\n")
    else:
        print("No misplaced tokens found.")
    
    print(f"\nAmount of Misplaced Tokens: {len(all_examples['misplaced'])}")
    
    TP = all_metrics["correct"]
    FN = all_metrics["missing"] + all_metrics["misplaced"]
    FP = all_metrics["false_positive"] + all_metrics["misplaced"]
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    F1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0
    
    print("\nOverall Evaluation Metrics:")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"F1 Score: {F1:.3f}")
    print(f"Accuracy: {accuracy:.3f}")
    
    print("\nPer-label Counts:")
    for label, counts in overall_label_metrics.items():
        print(f"{label}: {counts}")
    
    print("\nPer-label Evaluation Metrics:")
    for label, counts in overall_label_metrics.items():
        TP_label = counts["correct"]
        FN_label = counts["missing"] + counts["misplaced"]
        FP_label = counts["false_positive"] + counts["misplaced"]
        precision_label = TP_label / (TP_label + FP_label) if (TP_label + FP_label) > 0 else 0
        recall_label = TP_label / (TP_label + FN_label) if (TP_label + FN_label) > 0 else 0
        f1_label = 2 * precision_label * recall_label / (precision_label + recall_label) if (precision_label + recall_label) > 0 else 0
        print(f"{label} - Precision: {precision_label:.3f}, Recall: {recall_label:.3f}, F1: {f1_label:.3f}")
    
    return all_metrics, metrics_df, all_examples, overall_label_metrics

if __name__ == "__main__":
    # The file "dataset_1.csv" should contain the columns: original_text, labeled_text, masked_text.
    aggregate_metrics, per_row_metrics, examples, label_metrics = analyze_masking("dataset_1.csv")


Available columns in the DataFrame: ['original_text', 'labeled_text', 'masked_text']

Per-row masking metrics:
      correct  misplaced  missing  false_positive
0           2          0        0               1
1           3          0        0               0
2           1          0        0               0
3           2          0        0               0
4           2          0        0               0
...       ...        ...      ...             ...
4265        1          0        0               0
4266        1          0        0               0
4267        4          0        0               0
4268        1          0        0               0
4269        4          0        0               0

[4270 rows x 4 columns]

Aggregate masking metrics:
correct: 8792
misplaced: 40
missing: 100
false_positive: 73

Count of tokens with note 'No valid label found near mapped span': 95
Count of tokens with note 'Extra predicted token': 73

Examples of Misplaced Tokens:
Row 0 | Original Tex